# Disease Prediction from Symptoms

Map a set of symptoms to a likely disease. The data is the public Kaggle "Disease Symptom Prediction" table (41 diseases, 4,920 rows, up to 17 symptoms per row), loaded from a Hugging Face copy (`shanover/disease_symptoms_prec_full`, MIT). Each row is a disease plus a comma-separated symptom list.

One thing to know about this file: it is a disease-to-symptom reference table expanded into rows, not patient records. Most rows are exact repeats, so a random split puts the same rows in train and test and every model scores near 100%. I drop duplicate (disease, symptom set) rows before splitting so the test set only has symptom combinations the model hasn't seen.

Despite the folder name I'm doing this as **multi-class** (one label per row), not multi-label. The source data is single-label too.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(17)
DATA_DIR = Path("../../data/disease_symptoms")
PARQ = DATA_DIR / "disease_symptoms.parquet"
if not PARQ.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    src = ("https://huggingface.co/datasets/shanover/disease_symptoms_prec_full/"
           "resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet")
    pd.read_parquet(src).to_parquet(PARQ)
raw = pd.read_parquet(PARQ)
raw["symptom_set"] = raw["symptoms"].apply(lambda s: " ".join(sorted(t.strip() for t in s.split(",") if t.strip())))
df = raw.drop_duplicates(subset=["disease", "symptom_set"]).reset_index(drop=True)
print(f"raw rows: {len(raw)}  unique (disease, symptom set) rows: {len(df)}  diseases: {df.disease.nunique()}")
df[["disease", "symptom_set"]].head()

raw rows: 4920  unique (disease, symptom set) rows: 304  diseases: 41


,disease,symptom_set
0,Fungal infection,dischromic__patches itching nodal_skin_eruptio...
1,Fungal infection,dischromic__patches nodal_skin_eruptions skin_...
2,Fungal infection,dischromic__patches itching nodal_skin_eruptions
3,Fungal infection,dischromic__patches itching skin_rash
4,Fungal infection,itching nodal_skin_eruptions skin_rash


In [2]:
# each row's symptom set is already a space-separated string; CountVectorizer turns it into a multi-hot matrix
texts = df["symptom_set"].values
y = df["disease"].values

# some diseases have only 1-2 unique rows, so a stratified split is not possible for all of them
X_train, X_test, y_train, y_test = train_test_split(texts, y, test_size=0.2, random_state=17)

vec = CountVectorizer(token_pattern=r"[A-Za-z_()]+", binary=True, min_df=1)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)
print(f"vocab: {len(vec.vocabulary_)} symptoms, classes: {len(set(y))}, train: {len(X_train)}, test: {len(X_test)}")
print(f"test diseases never seen in train: {len(set(y_test) - set(y_train))}")

vocab: 131 symptoms, classes: 41, train: 243, test: 61
test diseases never seen in train: 0


## Two classifiers

In [3]:
clf_lr = LogisticRegression(max_iter=2000, C=2.0)
clf_lr.fit(Xtr, y_train)
pred_lr = clf_lr.predict(Xte)

clf_rf = RandomForestClassifier(n_estimators=300, random_state=17, n_jobs=2)
clf_rf.fit(Xtr, y_train)
pred_rf = clf_rf.predict(Xte)

print(f"LogReg accuracy:       {accuracy_score(y_test, pred_lr):.3f}")
print(f"RandomForest accuracy: {accuracy_score(y_test, pred_rf):.3f}")


LogReg accuracy:       1.000
RandomForest accuracy: 1.000


In [4]:
# spot-check: feed a custom symptom list and see what comes back
sample = "high_fever vomiting headache chills"
def predict(text, model=clf_lr, top_k=3):
    proba = model.predict_proba(vec.transform([text]))[0]
    top = np.argsort(proba)[-top_k:][::-1]
    return [(model.classes_[i], float(proba[i])) for i in top]

print(f"symptoms: {sample}")
for disease, p in predict(sample, top_k=5):
    print(f"  {disease:30s} {p:.3f}")


symptoms: high_fever vomiting headache chills
  Malaria                        0.190
  Paralysis (brain hemorrhage)   0.137
  Typhoid                        0.068
  Allergy                        0.048
  AIDS                           0.045


## Reality check

Even after removing the duplicate rows (4,920 rows collapse to 304 unique disease / symptom-set pairs), both models score 100% on the 61 held-out rows. That says more about the data than the models. Each disease in this file has its own fixed symptom list, and every row is just a subset of that list, so a couple of distinctive symptoms are usually enough to give the answer away. Without the dedup step the same 100% would also include memorized repeats, which is why I don't trust the near-perfect scores often quoted for this dataset.

The spot-check is more telling. "high_fever vomiting headache chills" gives Malaria at only 0.19 probability, with "Paralysis (brain hemorrhage)" in second place. Generic symptoms that many diseases share leave the model unsure, which is the realistic case.

A few caveats:

- This is a reference table, not patient records. Real medical data has comorbidities, missing symptoms, and overlapping presentations.
- I'm using multinomial classification, not multi-label. Adding a sigmoid-per-disease head would make this truly multi-label, which would also let it say "no condition matches strongly" instead of forcing a choice.
- **Do not treat this as a diagnostic tool.** This is an exercise in mapping discrete symptom sets to categorical outputs. A real triage system needs probability calibration, uncertainty estimates, and most importantly a human in the loop.